In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model and preprocess function

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

### Prepare the augmentation pipeline

In [ ]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [ ]:
print(ds['test'][0]['image'])

In [ ]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1] for pair in unique_pairs]

In [ ]:
len(r_class_names)

In [ ]:
augment_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)) # from clip
])

In [ ]:
def generate_N_views(N, transform_fn, image):
  views = [transform_fn(image) for _ in range(N)]
  curr_img = preprocess(image)
  views.append(curr_img)
  views = torch.stack(views)

  return views

In [ ]:
sample_views_for_idx_0 = generate_N_views(63, augment_transform, ds['test'][0]['image'])

In [ ]:
sample_views_for_idx_0.shape

### Training Loop

In [ ]:
from coop import PromptLearner, TextEncoderWrapper

In [ ]:
prompt_learner = PromptLearner(model, device, 4, tokenizer, 512, r_class_names)
text_encoder = TextEncoderWrapper(model)

for param in model.parameters():
  param.requires_grad_(False)

print([n for n, p in prompt_learner.named_parameters() if p.requires_grad])

In [ ]:
import torch.nn.functional as F

prompt_learner.reset_context()
optimizer = torch.optim.AdamW(prompt_learner.parameters(), lr=0.005)

test_image = ds['test'][0]['image']
test_image_views = generate_N_views(63, augment_transform, test_image)

image_features = model.encode_image(test_image_views.to(device))
prompts, tok_prompts = prompt_learner()
text_features = text_encoder(prompts, tok_prompts)
text_features = text_features / text_features.norm(dim=-1,keepdim=True)

logits = image_features @ text_features.t()
log_probs = F.log_softmax(logits, dim=-1)
probs = log_probs.exp()

per_view_entropy = -(probs * log_probs).sum(dim=-1)
k = max(1, int(0.1 * per_view_entropy.shape[0]))
top_confident = torch.topk(per_view_entropy, k=k, largest=False)
values, indices = top_confident

selected_probs = probs[indices]
avg_probs = selected_probs.mean(dim=0)
loss = -(avg_probs * avg_probs.log()).sum()

optimizer.zero_grad()
loss.backward()
optimizer.step()